# Topic 37 · DQN: CartPole, then LunarLander

CSCI 6379 Deep Learning · Fall 2026 · companion to [Reinforcement Learning, Part III](https://dongchul.kim/csci6379-fall2026/topic/37) and its [slides](https://dongchul.kim/csci6379/slides/topic37.pdf).

One DQN, about 100 lines of PyTorch: a $Q$-network, a replay buffer, a target network, $\varepsilon$-greedy exploration.
It learns **CartPole** in a few minutes, then, with only its settings changed, **LunarLander**.

**A CPU runtime is fine** (the network is a small MLP; a GPU does not make it faster). **Runtime > Run all**.
CartPole takes about 2 to 4 minutes, LunarLander about 5 to 15. Your numbers will differ a little from the slides.

## Setup

Gymnasium has both games; LunarLander needs the Box2D physics engine, which needs `swig` to build.

In [ ]:
!pip install -q swig
!pip install -q "gymnasium[box2d]" imageio

In [ ]:
import collections, copy, random, time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn

## The settings

Only `CONFIG` differs between the two games. `test_every`: how often we test the greedy agent (and keep the best network).

In [ ]:
CONFIG = {
    "CartPole-v1":    dict(hidden=128, lr=5e-4, buffer=50_000,  batch=64,  steps=100_000,
                           eps_steps=10_000, learn_start=1_000, train_every=1, target_every=500,
                           test_every=2_500, solved=500),
    "LunarLander-v3": dict(hidden=256, lr=2.5e-4, buffer=100_000, batch=128, steps=500_000,
                           eps_steps=50_000, learn_start=5_000, train_every=4, target_every=1_000,
                           test_every=10_000, solved=250),
}
GAMMA, EPS_END, TESTS = 0.99, 0.05, 5

## The $Q$-network

A state goes in, one $Q$-value per action comes out: the table of Q-learning, replaced by an MLP.

In [ ]:
class QNet(nn.Module):
    """A state goes in; one Q-value per action comes out."""
    def __init__(self, n_obs, n_act, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_act))

    def forward(self, s):
        return self.net(s)

## The replay buffer

Keep the last transitions; train on a random sample of them, so neighbouring steps rarely share a batch.

In [ ]:
class Replay:
    """The last `size` transitions; a batch is a random sample."""
    def __init__(self, size):
        self.data = collections.deque(maxlen=size)

    def push(self, s, a, r, s2, done):
        self.data.append((s, a, r, s2, done))

    def sample(self, n):
        s, a, r, s2, done = zip(*random.sample(self.data, n))
        f = lambda x: torch.tensor(np.array(x), dtype=torch.float32)
        return f(s), torch.tensor(a), f(r), f(s2), f(done)

## Training

The whole algorithm. Read it in five parts (the slides show each one):
the two networks (`q` and the frozen `q_target`), acting with $\varepsilon$-greedy, learning from a random batch with
the target $r + \gamma \max_{a'} Q'(s',a')$, copying `q` into `q_target`, and keeping the best network.
`play` runs greedy episodes; `train` uses it for the tests.

In [ ]:
def play(q, env_name, episodes=1, seed=100, frames=None):
    """Greedy episodes (q=None: random actions). Returns the episode returns."""
    env = gym.make(env_name, render_mode="rgb_array" if frames is not None else None)
    out = []
    for e in range(episodes):
        s, _ = env.reset(seed=seed + e)
        total, done = 0.0, False
        while not done:
            if frames is not None and e == 0:
                frames.append(env.render())
            if q is None:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = q(torch.tensor(s, dtype=torch.float32)).argmax().item()
            s, r, terminated, truncated, _ = env.step(a)
            total += r
            done = terminated or truncated
        out.append(total)
    return out

In [ ]:
def train(env_name, seed=0, log=print):
    c = CONFIG[env_name]
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    env = gym.make(env_name)
    n_obs, n_act = env.observation_space.shape[0], env.action_space.n

    # [slide:setup]
    q = QNet(n_obs, n_act, c["hidden"])                  # the network we train
    q_target = QNet(n_obs, n_act, c["hidden"])           # a frozen copy, for the targets
    q_target.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=c["lr"])
    memory = Replay(c["buffer"])
    # [/slide]

    returns, ep_ret = [], 0.0
    best, best_state, evals = -np.inf, None, []
    s, _ = env.reset(seed=seed)
    t0 = time.time()
    for step in range(1, c["steps"] + 1):
        # [slide:act]
        eps = max(EPS_END, 1 - step / c["eps_steps"])    # 1.0 down to 0.05
        if random.random() < eps:
            a = env.action_space.sample()                # explore
        else:
            with torch.no_grad():
                a = q(torch.tensor(s, dtype=torch.float32)).argmax().item()   # exploit
        s2, r, terminated, truncated, _ = env.step(a)
        memory.push(s, a, r, s2, terminated)
        s, ep_ret = s2, ep_ret + r
        # [/slide]
        if terminated or truncated:
            returns.append(ep_ret)
            if len(returns) % 20 == 0:
                log(f"episode {len(returns):4d}  step {step:7d}  eps {eps:.2f}  "
                    f"last 20 average {np.mean(returns[-20:]):7.1f}  {time.time() - t0:5.0f} s")
            s, _ = env.reset()
            ep_ret = 0.0

        if step >= c["learn_start"] and step % c["train_every"] == 0:
            # [slide:learn]
            S, A, R, S2, D = memory.sample(c["batch"])
            with torch.no_grad():                         # the target: r + gamma * max Q'(s', a')
                y = R + GAMMA * (1 - D) * q_target(S2).max(dim=1).values
            q_sa = q(S).gather(1, A.unsqueeze(1)).squeeze(1)   # Q(s, a) for the actions taken
            loss = ((q_sa - y) ** 2).mean()               # the squared TD error
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(q.parameters(), 10)
            opt.step()
            # [/slide]
        if step % c["target_every"] == 0:
            q_target.load_state_dict(q.state_dict())      # refresh the frozen copy
        # [slide:keep]
        if step % c["test_every"] == 0:              # every few thousand steps,
            score = np.mean(play(q, env_name, episodes=TESTS))   # test the greedy agent
            evals.append((step, score))
            if score > best:                         # keep the best network so far
                best, best_state = score, copy.deepcopy(q.state_dict())
            log(f"  test at step {step}: {score:.1f}  (best {best:.1f})")
            if score >= c["solved"]:                 # good enough: stop
                break
        # [/slide]
    q.load_state_dict(best_state)
    return q, returns, evals

In [ ]:
import matplotlib.pyplot as plt

def plot(returns, evals, title, solved):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    r = np.array(returns, float)
    ax[0].plot(r, color="#C7D2FE", lw=0.8, label="each episode")
    ax[0].plot([r[max(0, i - 19): i + 1].mean() for i in range(len(r))], color="#4338CA", lw=2, label="last 20, average")
    ax[0].axhline(solved, color="#D97706", ls="--"); ax[0].set_xlabel("episode"); ax[0].set_ylabel("return"); ax[0].legend()
    st, sc = zip(*evals)
    ax[1].plot(st, sc, "o-", color="#4338CA"); ax[1].axhline(solved, color="#D97706", ls="--")
    ax[1].set_xlabel("training step"); ax[1].set_ylabel("greedy test score")
    fig.suptitle(title); plt.show()

import imageio
from IPython.display import Image, display

def show(agent, env_name, path):
    """Play one greedy episode (agent=None: random actions), save it as a GIF and show it."""
    frames = []
    ret = play(agent, env_name, episodes=1, frames=frames)[0]
    imageio.mimsave(path, frames[::2], duration=1 / 25, loop=0)
    print(f"{path}: return {ret:.0f}, {len(frames)} steps")
    display(Image(filename=path))

## 1. CartPole

4 numbers in, 2 actions out, $+1$ for every step the pole stays up, 500 at most.

In [ ]:
t0 = time.time()
q_cp, returns_cp, evals_cp = train("CartPole-v1", seed=0)
print(f"{time.time() - t0:.0f} s")
plot(returns_cp, evals_cp, "CartPole-v1", 500)

Before and after: an untrained (random) agent, then the trained network.

In [ ]:
show(None, "CartPole-v1", "cartpole_before.gif")
show(q_cp, "CartPole-v1", "cartpole_after.gif")

## 2. LunarLander

8 numbers in, 4 actions out (nothing, left engine, main engine, right engine). Flying to the pad and slowing down earns about $+100$ to $+140$, coming to rest $+100$ more, a crash costs $-100$; 200 counts as solved. **The same `train` function**, with LunarLander's `CONFIG`.

In [ ]:
t0 = time.time()
q_ll, returns_ll, evals_ll = train("LunarLander-v3", seed=0)
print(f"{time.time() - t0:.0f} s")
plot(returns_ll, evals_ll, "LunarLander-v3", 200)

In [ ]:
show(None, "LunarLander-v3", "lunar_before.gif")
show(q_ll, "LunarLander-v3", "lunar_after.gif")

## Try this

Change one thing, run the CartPole cells again, and compare the curve:

- `GAMMA = 0.9` instead of 0.99: the agent becomes short-sighted.
- Set `target_every=1` in CartPole's `CONFIG`: no frozen target network.
- `buffer=500`: almost no replay. Or `eps_steps=1`: almost no exploration.
- A different `seed`: same code, a different run.

Back to the [Topic 37 lecture note](https://dongchul.kim/csci6379-fall2026/topic/37).